In [29]:
import os.path as osp
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, BoundedIntText, HBox, Layout, jslink, interactive_output
from IPython.display import display
from scipy.signal import welch

In [30]:
datadir = Path("~/Desktop/UCSDPDCv2/data/DSumTrig_22V/").expanduser()
files = list(datadir.glob("**/**.csv"))

print(f"Running {len(files)} files")

Running 2 files


In [31]:
def read_data(file):
    data = pd.read_csv(file, delimiter=";")
    # Ceate a 2D array for each PDC
    pdc_split = {pdc: pdc_data.groupby(["frameIdx", "dataIdx"])["dsum"].sum().unstack(fill_value=0).to_numpy() for pdc, pdc_data in data.groupby("pdcIdx")}

    return pdc_split

def averaged_plots(file):
    split_data = read_data(file)

    fig, ax = plt.subplots(2,1,figsize=(8,6))
    for pdc, data in split_data.items():
        avg = np.mean(data, axis=1)
        ax[0].plot(avg, label=f"PDC {pdc}")
        avg = np.mean(data, axis=0)
        times_us = [x*.01 for x in range(len(avg))]
        ax[1].plot(times_us, avg, label=f"PDC {pdc}")
        
    ax[0].set_title("Average DSum Counts per Frame")
    ax[0].set_xlabel("Frame")
    ax[1].set_title("Average DSum Counts")
    ax[1].set_xlabel("Time (us)")

    for axis in ax:
        axis.set_ylabel("Counts")
        axis.set_yscale("log")
        axis.legend()
    fig.tight_layout()
    savepath = file.parent / "AveragedCounts.png"
    fig.savefig(savepath)
    plt.close(fig)

def PSD(file):
    split_data = read_data(file)

    plt.figure(figsize=(8, 3))
    for pdc, arr in split_data.items():
        f, p = welch(arr, fs=1e8, nperseg=100, window="hann", detrend="constant", axis=1)
        plt.plot(f / 1e6, p.mean(axis=0), label=f"PDC {pdc}")
    plt.title("Frame-averaged PSD")
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("PSD (dsum² / Hz)")
    plt.yscale("log")
    plt.legend()
    plt.tight_layout()   

    savepath = file.parent / "PSD.png"
    plt.savefig(savepath)
    plt.close()

### Frame and Time averaged Counts

In [32]:
for file in files:
    averaged_plots(file)

### Frame averaged PSD

In [33]:
for file in files:
    PSD(file)

### Interactive Per-Frame Plotters

In [ ]:
file = files[0]
data = read_data(file)
frame_num = len(next(iter(data.values())))

clock_speed_ns = 10
def show(frame):
    plt.figure(figsize=(8, 3))
    for pdc, arr in data.items():
        if frame < len(arr):
            t_us = np.arange(arr.shape[1]) * clock_speed_ns / 1e3
            plt.plot(t_us, arr[frame], label=f"PDC {pdc}")
    plt.title(f"Frame {frame}")
    plt.xlabel("Time (µs)")
    plt.ylabel("DSum")
    plt.yscale("log")
    plt.legend()
    plt.show()


def show_psd(frame):
    plt.figure(figsize=(8, 3))
    for pdc, arr in data.items():
        if frame < len(arr):
            f, p = welch(arr[frame], fs=1e9/clock_speed_ns, nperseg=100, window="hann", detrend="constant")
            plt.plot(f / 1e6, p, label=f"PDC {pdc}")
    plt.title(f"Frame {frame}: PSD")
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("PSD (dsum² / Hz)")
    plt.yscale("log")
    plt.legend()
    plt.show()


def frame_browser(func):
    """Slider + text box linked together, driving func(frame)."""
    slider = IntSlider(min=0, max=frame_num - 1, continuous_update=False, layout=Layout(width="700px"))
    box = BoundedIntText(min=0, max=frame_num - 1, layout=Layout(width="90px"))
    jslink((slider, "value"), (box, "value"))
    display(HBox([slider, box]), interactive_output(func, {"frame": slider}))


frame_browser(show)        # raw dsum vs. index
frame_browser(show_psd)    # PSD

Output()

Output()